Below is a practical, end-to-end recipe you can adapt to your own cutting-test data. I assume your raw table looks roughly like this:

| feed_mm_min | spindle_rpm | depth_ap_mm | edges_z | ae_pct |
|-------------|-------------|-------------|---------|--------|
| 1200        | 6500        | 1.0         | 2       | 25     |
| ...         | ...         | ...         | ...     | 95     |

1. Decide: classification vs regression
Because ae only takes four discrete values (25 / 50 / 75 / 95 %), a 4-class classifier is simplest and lets you monitor class-level performance (confusion matrix).
If in a later study you allow any ae between 0 – 100 %, switch to a regression loss (e.g. MSE).

2. Python environment
bash
Copiar
Editar
pip install pandas scikit-learn scikeras[tensorflow] matplotlib seaborn
scikeras is a thin wrapper that lets you treat a Keras model as if it were an sklearn estimator—handy for pipelines and tuning.

3. Pre-processing pipeline
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from scikeras.wrappers import KerasClassifier
from tensorflow import keras



In [ ]:
# ------------------------------------------------------------------
df = pd.read_csv("cutting_tests.csv")

X = df.drop("ae_pct", axis=1)
y = df["ae_pct"].astype("category")          # 25/50/75/95

num_cols = ["feed_mm_min", "spindle_rpm", "depth_ap_mm"]
cat_cols = ["edges_z"]                       # treat z as categorical

preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(), cat_cols),
    ]
)



Why scale? The MLP optimiser converges faster when inputs are roughly N(0, 1).
Why one-hot edges_z? A 3-edge cutter is not “1.5 × better” than a 2-edge cutter.


In [ ]:
# 4. Define an MLP architecture

def build_model(n_hidden=2, n_neurons=32, dropout=0.1, learning_rate=1e-3):
    model = keras.Sequential()
    model.add(keras.layers.Input(shape=(X.shape[1],)))

    for _ in range(n_hidden):
        model.add(keras.layers.Dense(n_neurons, activation="relu"))
        model.add(keras.layers.Dropout(dropout))

    model.add(keras.layers.Dense(4, activation="softmax"))  # 4 classes

    model.compile(
        loss="categorical_crossentropy",
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        metrics=["accuracy"],
    )
    return model
# 5. Wrap it in an sklearn pipeline

clf = Pipeline(
    steps=[
        ("prep", preprocess),
        ("net", KerasClassifier(
            model=build_model,
            epochs=200,
            batch_size=32,
            verbose=0,
            callbacks=[keras.callbacks.EarlyStopping(patience=15, restore_best_weights=True)]
        )),
    ]
)
# 6. Hyper-parameter search (optional but recommended)
from sklearn.model_selection import RandomizedSearchCV
import scipy.stats as st

param_distribs = {
    "net__n_hidden": st.randint(1, 4),
    "net__n_neurons": st.randint(16, 128),
    "net__dropout": st.uniform(0.05, 0.3),
    "net__learning_rate": st.loguniform(1e-4, 5e-3),
}

search = RandomizedSearchCV(
    clf,
    param_distributions=param_distribs,
    n_iter=30,
    cv=3,
    scoring="accuracy",
    n_jobs=-1,
    refit=True,
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

search.fit(X_train, y_train)
print("Best CV accuracy:", search.best_score_)
Early stopping keeps you from over-fitting; the random search scopes out a reasonable architecture in ~30 fits.

7. Evaluate
python
Copiar
Editar
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

y_pred = search.predict(X_test)

print(classification_report(y_test, y_pred, digits=3))

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, cmap="Blues", labels=[25, 50, 75, 95]
)
plt.title("ae% confusion matrix")
plt.show()
Key checks:

Overall accuracy > baseline (25 %)?

Per-class recall – Are 25 % and 95 % cuts harder to recognise?

Confusion trends – If 75 % is often mistaken for 50 %, maybe feed or depth ranges overlap.

8. Tips for better performance
Issue	Remedies
Few samples per class	k-fold CV, data augmentation (add Gaussian noise within sensor tolerance)
Sensor noise / different machines	Add machine ID or sensor channel as categorical feature
Strong physical relations	Include chip-load = feed / (edges_z·spindle_rpm) as an extra feature
Real-time need	After training, export model via tf.keras.models.save_model(); deploy in C++ using TF-Lite or ONNX

9. Regression variant (if you want continuous ae)
Just change the last layer to Dense(1, activation="linear") and use loss="mse" with metrics like MAE or R². In that case you can treat ae as a float in [0, 1] (normalise by dividing by 100).

Final checklist before you hit “Run”
Data sanity: no weird units, no outliers (e.g., 12 000 rpm typed as 120 000).

Balanced classes: a quick y.value_counts() to confirm.

Document every step – process engineers will ask for traceability.

Feel free to paste your real dataset header or any specific roadblocks, and we can iterate on the exact code.








Tú dijiste:
Ia have several csv files for each combination


ChatGPT Plus

Thought for 6 seconds
Below is a pragmatic pattern you can copy-paste (and tweak) to turn “lots of small CSVs—one per cutting-parameter combo” into a single, tidy dataframe ready for the neural-network pipeline I sketched earlier.

1 Organise the files
Put every CSV in one folder—say ./data_raw/.

Copiar
Editar
data_raw/
├── feed1200_rpm6500_ap1_z2.csv
├── feed1200_rpm6500_ap1_z3.csv
├── …
└── feed2400_rpm8000_ap2_z4.csv
Tip: Encode the key parameters in the filename with a consistent pattern; that lets you recover them even if the file’s internal columns don’t repeat the metadata.

2 Read & stack with glob + pandas
python
Copiar
Editar
import glob, re, pandas as pd

def parse_filename(fname):
    """
    Extract process parameters from a name like
    'feed1200_rpm6500_ap1_z2.csv'
    """
    m = re.search(
        r"feed(?P<feed>\d+)_rpm(?P<rpm>\d+)_ap(?P<ap>[\d.]+)_z(?P<z>[234])",
        fname
    )
    if not m:
        raise ValueError(f"Unrecognised filename: {fname}")
    return {k: float(v) if k != "z" else int(v) for k, v in m.groupdict().items()}


dfs = []

for f in glob.glob("data_raw/*.csv"):
    df_tmp = pd.read_csv(f)

    # ① add columns extracted from the filename
    meta = parse_filename(f)
    for k, v in meta.items():
        df_tmp[k] = v

    # ② OPTIONAL: keep track of which file/run the rows came from
    df_tmp["run_id"] = f.split("/")[-1].replace(".csv", "")

    dfs.append(df_tmp)

df = pd.concat(dfs, ignore_index=True)
What if the CSV already contains the parameters inside?
Then you can skip the regex and simply

python
Copiar
Editar
df_tmp["run_id"] = f  # or any unique identifier
and still ­concat them.

3 Sanity-check the aggregate table
python
Copiar
Editar
print(df.head())
print(df.describe(include="all"))
print(df["ae_pct"].value_counts())
Make sure you see:

feed, rpm, ap, z, ae_pct as separate columns.

No weird NaNs or duplicate headings.

4 Save the master dataset (optional but recommended)
python
Copiar
Editar
df.to_csv("cutting_tests_master.csv", index=False)
Now you have one canonical file you can version-control (e.g. Git-LFS or DVC) and feed straight into the preprocessing / MLP pipeline:

python
Copiar
Editar
df = pd.read_csv("cutting_tests_master.csv")
X = df[["feed", "rpm", "ap", "z"]]
y = df["ae_pct"]
5 Handling very large datasets
If the raw files are huge (gigabytes):

Stream in chunks

python
Copiar
Editar
for chunk in pd.read_csv(f, chunksize=100_000):
    …
Parquet – Convert once with df.to_parquet() for faster IO and compression.

Dask / Polars – Swap pandas for a parallel dataframe library.

6 Common pitfalls & quick fixes
Symptom	Likely cause	Quick fix
Duplicate rows	Same file concatenated twice	Keep a list of processed filenames.
ae_pct read as string	Comma decimal separator or trailing “%”	pd.read_csv(..., decimal=',') or df["ae_pct"] = df["ae_pct"].str.rstrip('%').astype(float)
Memory blow-up	Tens of millions of rows	Use dtype hints, chunked reading, or out-of-core tools.

Next step
Once df looks good, run the preprocessing / neural-network code from the previous message. If you hit any specific errors—parsing, memory, model convergence—feel free to paste the traceback or describe the behaviour and we’ll troubleshoot the next layer.








Tú dijiste:
I want to Identity values for ae that are between the measurements. E.g. if the clasification results in 0.7 for 25% and 0.3 for 50% I would clasify the value as 0.25*.7 + .5*.3


ChatGPT Plus

Thought for 9 seconds
Turning class-probabilities into a continuous ae prediction
A very handy trick is to keep the soft-max classification head unchanged and, at inference time, collapse the class-probabilities into a single scalar by taking their expectation:

𝑎
^
𝑒
  
=
  
∑
𝑘
=
1
4
𝑝
𝑘
 
𝑎
𝑒
,
𝑘
(
where 
𝑎
𝑒
,
𝑘
∈
{
0.25
,
0.50
,
0.75
,
0.95
}
)
a
^
  
e
​
 = 
k=1
∑
4
​
 p 
k
​
 a 
e,k
​
 (where a 
e,k
​
 ∈{0.25,0.50,0.75,0.95})
Your example becomes

makefile
Copiar
Editar
p = [0.70, 0.30, 0.00, 0.00]
a = [0.25, 0.50, 0.75, 0.95]
np.dot(p, a)  # 0.325  (= 32.5 %)
The network still learns to separate the four regimes, but the post-processing gives you in-between values for free.

1 Small tweak to the training labels
Keep the classification labels as 25/50/75/95 for fitting but store a float copy (0.25–0.95) for regression-style metrics:

python
Copiar
Editar
y_cls  = df["ae_pct"].astype("category")      # 25 / 50 / 75 / 95
y_cont = df["ae_pct"] / 100.0                 # 0.25 … 0.95
You will train on y_cls exactly as before; y_cont is only for evaluation later.

2 Fit exactly the same pipeline
No change to the model definition or the Pipeline.
(It is still a 4-way classifier with a soft-max output.)

3 Collapse probabilities → continuous prediction
python
Copiar
Editar
import numpy as np
from sklearn.metrics import mean_absolute_error, r2_score

LABEL_VALUES = np.array([0.25, 0.50, 0.75, 0.95])  # shape (4,)

# --- get class-probabilities on the test fold
proba = search.predict_proba(X_test)               # shape (n_samples, 4)

# --- turn them into a single scalar per sample
ae_pred = proba @ LABEL_VALUES                     # dot-product

mae = mean_absolute_error(y_cont_test, ae_pred)
r2  = r2_score(y_cont_test, ae_pred)
print(f"MAE  : {mae:.4f}  (in fraction of tool width)")
print(f"R²   : {r2:.3f}")
If you prefer percentages, multiply everything by 100.

4 Why this works (and when it doesn’t)
Ordinal structure is respected.
The network’s logits learn a smooth energy landscape; adjacent classes tend to blend in physically plausible ways.

Bounded output.
Prediction can never go below 25 % or above 95 %.
If you one day drill experiments at, say, 10 % or 100 % engagement, simply add those classes and the expectation formula still works.

No extra training.
You avoid switching losses, re-tuning hyper-parameters, or worrying about skewed regression targets.

Watch out: If classes are extremely imbalanced, the soft-max may put too much mass on the majority class; calibrate with techniques like temperature scaling or focal loss if needed.

5 Alternatives you can try later
Approach	Pros	Cons
Pure regression head
(Dense(1, "linear"), MSE loss)	Directly optimises MAE/MSE; predictions not range-limited	Harder to learn when labels are clustered; may predict <25 % or >95 %
Ordinal (cumulative) regression	Exploits order without forcing equal spacing	Requires custom loss layer
Multi-task – one soft-max and one regression output	Can share features while giving both signals	Slightly more code and tuning

For your stated goal—just interpolate between measured values—the expected-value trick is usually the fastest route to good numbers.

Drop-in code block for your current project
python
Copiar
Editar
# after you've fitted `search` (RandomizedSearchCV) and have X_test / y_test
proba_test = search.predict_proba(X_test)          # (n, 4)
ae_pred_pct = (proba_test @ np.array([25, 50, 75, 95]))  # in percent
ae_true_pct = y_cls_test.astype(float)

print("MAE (%):", mean_absolute_error(ae_true_pct, ae_pred_pct))